In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


PATH = "selected_laptop_features_dataset.csv"


def print_section(title: str):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)


def print_metrics(y_true, y_pred, title):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)
    print(f"MSE:  {mse:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"MAE:  {mae:.2f}")
    print(f"R2:   {r2:.4f}")


# ============================================
# ВЕРСИЯ 1 MLP 
# =============================================

def library_mlp_regressor(df: pd.DataFrame):
    print_section("ВЕРСИЯ 1. MLP-регрессор")

    X = df.drop("Price", axis=1)
    y = df["Price"]

    categorical_features = [c for c in ["Brand"] if c in X.columns]

    numeric_features = [
        c for c in X.columns
        if c not in categorical_features
    ]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

    try:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_features),
            ("cat", encoder, categorical_features)
        ]
    )

    model = MLPRegressor(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        solver="adam",
        learning_rate_init=0.001,
        max_iter=1000,
        random_state=42,
        early_stopping=True
    )

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    y_scaler = StandardScaler()

    y_train_scaled = y_scaler.fit_transform(
        y_train.to_numpy().reshape(-1, 1)
    ).ravel()

    pipeline.fit(X_train, y_train_scaled)

    y_pred_scaled = pipeline.predict(X_test)

    y_pred = y_scaler.inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    ).ravel()

    print_metrics(
        y_test,
        y_pred,
        "Метрики качества библиотечной модели"
    )

    result = pd.DataFrame({
        "Real Price": y_test.to_numpy(),
        "Predicted Price": y_pred
    })

    print("\nПервые 10 предсказаний библиотечной модели:")
    print(result.head(10))

    trained_model = pipeline.named_steps["model"]

    if hasattr(trained_model, "loss_curve_"):
        plt.figure(figsize=(8, 5))
        plt.plot(trained_model.loss_curve_)
        plt.xlabel("Эпоха")
        plt.ylabel("Loss")
        plt.title("График ошибки обучения sklearn MLPRegressor")
        plt.grid(True)
        plt.tight_layout()
        plt.show()

    return pipeline

df = pd.read_csv(PATH)

print_section("Загрузка обработанного датасета")
print("Shape:", df.shape)
print(df.head())
print(df.dtypes)

library_model = library_mlp_regressor(df)
